In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    precision_recall_curve,
    classification_report,
)

In [2]:
train_df = pd.read_csv("../data/processed/train_features.csv")
test_df = pd.read_csv("../data/processed/test_features.csv")

# Ensure date is datetime
train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])

# Quick check
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns sample:")
print(train_df.columns[:10].tolist())

Train shape: (1692, 170)
Test shape: (564, 169)

Train columns sample:
['date', '5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit']


In [3]:
train_df.head(5)

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,...,sst_cameroon_mean_temperature_deg_c_lag_1,sst_cameroon_mean_temperature_deg_c_lag_7,sst_cameroon_mean_temperature_deg_c_lag_14,sst_cameroon_mean_temperature_deg_c_diff_1,sst_cameroon_mean_temperature_deg_c_diff_7,sst_indian_ocean_mean_temperature_deg_c_lag_1,sst_indian_ocean_mean_temperature_deg_c_lag_7,sst_indian_ocean_mean_temperature_deg_c_lag_14,sst_indian_ocean_mean_temperature_deg_c_diff_1,sst_indian_ocean_mean_temperature_deg_c_diff_7
0,2002-07-30,0.2568,21.1858,32.7137,101.0253,25.479012,0.23246,28.102166,0.15961,2.4782,...,25.469736,25.287126,25.534091,0.009276,0.191886,27.979835,28.165013,28.571528,0.122331,-0.062847
1,2002-07-31,0.2155,19.0670,32.3881,100.9851,25.480693,0.24430,27.982893,0.15391,2.6832,...,25.479012,25.344692,25.510941,0.001681,0.136001,28.102166,28.120344,28.395926,-0.119273,-0.137450
2,2002-08-01,0.2189,20.1212,35.3176,100.9979,25.413633,0.23324,27.992489,0.12183,-3.4738,...,25.480693,25.407062,25.411424,-0.067060,0.006571,27.982893,28.166922,28.337341,0.009595,-0.174433
3,2002-08-02,0.2531,20.6066,32.6913,101.1590,25.341496,0.30679,27.956082,0.12166,1.4265,...,25.413633,25.445490,25.321285,-0.072137,-0.103994,27.992489,27.972588,28.326524,-0.036407,-0.016506
4,2002-08-03,0.2743,19.4863,33.6305,101.0562,25.152698,0.31285,27.982394,0.11980,5.7806,...,25.341496,25.483519,25.233598,-0.188797,-0.330821,27.956082,27.902828,28.324255,0.026312,0.079566


In [5]:
test_df.head()

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,...,sst_cameroon_mean_temperature_deg_c_lag_1,sst_cameroon_mean_temperature_deg_c_lag_7,sst_cameroon_mean_temperature_deg_c_lag_14,sst_cameroon_mean_temperature_deg_c_diff_1,sst_cameroon_mean_temperature_deg_c_diff_7,sst_indian_ocean_mean_temperature_deg_c_lag_1,sst_indian_ocean_mean_temperature_deg_c_lag_7,sst_indian_ocean_mean_temperature_deg_c_lag_14,sst_indian_ocean_mean_temperature_deg_c_diff_1,sst_indian_ocean_mean_temperature_deg_c_diff_7
0,2020-07-30,0.3543,21.0362,33.6306,101.2622,25.081599,0.20056,29.087824,0.13038,-3.5140,...,25.098329,26.020651,25.278619,-0.016730,-0.939052,29.017544,29.609844,29.750814,0.070280,-0.522020
1,2020-07-31,0.3725,21.4910,27.2711,101.3073,25.082625,0.19472,29.209763,0.12066,0.1880,...,25.081599,25.880483,25.559518,0.001026,-0.797858,29.087824,29.591446,29.799926,0.121939,-0.381683
2,2020-08-01,0.3739,19.5870,27.2990,101.1454,25.333277,0.19720,29.229695,0.12358,3.4191,...,25.082625,25.682720,25.687322,0.250652,-0.349443,29.209763,29.579634,29.787704,0.019933,-0.349938
3,2020-08-02,0.3556,20.2491,30.5264,101.1783,25.422197,0.19240,29.150845,0.12556,0.2981,...,25.333277,25.470760,25.748548,0.088920,-0.048563,29.229695,29.402182,29.759672,-0.078850,-0.251337
4,2020-08-03,0.3545,21.2719,29.0640,101.3373,25.464339,0.19378,29.002645,0.12146,-2.3859,...,25.422197,25.198451,25.976848,0.042142,0.265888,29.150845,29.049885,29.573546,-0.148201,-0.047241


In [6]:
target_col = "dryspell_warn_7d"

drop_cols = ["date", "season_year", target_col]

if "year" in train_df.columns:
    drop_cols.append("year")

feature_cols = [col for col in train_df.columns if col not in drop_cols]
print("Number of features:", len(feature_cols))
print(feature_cols[:20])

Number of features: 166
['5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit', '2m_temp', 'u10', 'vapor_pressure_deficit', 'month', '5cm_soil_moist_rollmean_7', '5cm_soil_moist_rollmin_7', '5cm_soil_moist_rollmax_7', '5cm_soil_moist_rollmean_14', '5cm_soil_moist_rollmin_14', '5cm_soil_moist_rollmax_14', '5cm_soil_moist_rollmean_30']


In [7]:
train_mask =train_df["season_year"]<= 2016
val_mask =train_df["season_year"]>= 2017

train_part = train_df.loc[train_mask].copy()
val_part = train_df.loc[val_mask].copy()

X_train = train_part[feature_cols]
y_train = train_part[target_col]

X_val = val_part[feature_cols]
y_val = val_part[target_col]

print("Train seasons:", sorted(train_part["season_year"].unique()))
print("Validation seasons:", sorted(val_part["season_year"].unique()))
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

Train seasons: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
Validation seasons: [np.int64(2017), np.int64(2018), np.int64(2019)]
X_train shape: (1410, 166)
X_val shape: (282, 166)


In [8]:
models = {
    "logreg":Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "hist_gb": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter= 300,
        random_state=42
    )
}